In [1]:
import scanpy as sc
atac_peak = sc.read_h5ad(r"D:\Scunpair_Project\Dataset\scRNA+scATAC 10x-Multiome-Pbmc10k\10x-Multiome-Pbmc10k-ATAC.h5ad")

In [2]:
###layer.py
import math
import numpy as np

import torch
from torch import nn as nn
import torch.nn.functional as F
import torch.autograd as autograd
from torch.distributions import Normal
from torch.nn.parameter import Parameter
from torch.nn import init
from torch.autograd import Function


activation = {
    'relu':nn.ReLU(),
    'rrelu':nn.RReLU(),
    'sigmoid':nn.Sigmoid(),
    'leaky_relu':nn.LeakyReLU(),
    'tanh':nn.Tanh(),
    'gelu':nn.GELU(),
    'softmax':nn.Softmax(dim=1),
    '':None
}


class DSBatchNorm(nn.Module):
    """
    Domain-specific Batch Normalization
    """
    def __init__(self, num_features, n_domain, eps=1e-5, momentum=0.1):
        """
        Parameters
        ----------
        num_features
            dimension of the features
        n_domain
            domain number
        """
        super().__init__()
        self.n_domain = n_domain
        self.num_features = num_features
        self.bns = nn.ModuleList([nn.BatchNorm1d(num_features, eps=eps, momentum=momentum) for i in range(n_domain)])
        
    def reset_running_stats(self):
        for bn in self.bns:
            bn.reset_running_stats()
            
    def reset_parameters(self):
        for bn in self.bns:
            bn.reset_parameters()
            
    def _check_input_dim(self, input):
        raise NotImplementedError
    #y int       
    def forward(self, x, y):
        out = torch.zeros(x.size(0), self.num_features, device=x.device) #, requires_grad=False)
        
        out = self.bns[y](x)
            
        return out
        

class Block(nn.Module):
    """
    Basic block consist of:
        fc -> bn -> act -> dropout
    """
    def __init__(
            self,
            input_dim, 
            output_dim, 
            norm='', 
            act='', 
            dropout=0
        ):
        """
        Parameters
        ----------
        input_dim
            dimension of input
        output_dim
            dimension of output
        norm
            batch normalization, 
                * '' represent no batch normalization
                * 1 represent regular batch normalization
                * int>1 represent domain-specific batch normalization of n domain
        act
            activation function,
                * relu -> nn.ReLU
                * rrelu -> nn.RReLU
                * sigmoid -> nn.Sigmoid()
                * leaky_relu -> nn.LeakyReLU()
                * tanh -> nn.Tanh()
                * '' -> None
        dropout
            dropout rate
        """
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)
        nn.init.xavier_uniform_(self.fc.weight)
        if type(norm) == int:
            if norm==1: # TO DO
                self.norm = nn.BatchNorm1d(output_dim)
            else:
                self.norm = DSBatchNorm(output_dim, norm)
        else:
            self.norm = None
            
        self.act = activation[act]
            
        if dropout >0:
            self.dropout = nn.Dropout(dropout)
        else:
            self.dropout = None
            
    def forward(self, x, y=None):
        h = self.fc(x)
        if self.norm:
            if len(x) == 1:
                pass
            elif self.norm.__class__.__name__ == 'DSBatchNorm':
                h = self.norm(h, y)
            else:
                h = self.norm(h)
        if self.act:
            h = self.act(h)
        if self.dropout:
            h = self.dropout(h)
        return h
        
    
class NN(nn.Module):
    """
    Neural network consist of multi Blocks
    """
    def __init__(self, input_dim, cfg):
        """
        Parameters
        ----------
        input_dim
            input dimension
        cfg
            model structure configuration, 'fc' -> fully connected layer
            
        Example
        -------
        >>> latent_dim = 10
        >>> dec_cfg = [['fc', x_dim, n_domain, 'sigmoid']]
        >>> decoder = NN(latent_dim, dec_cfg)
        
        
        """
        super().__init__()
        net = []
        for i, layer in enumerate(cfg):
            if i==0:
                d_in = input_dim
            if layer[0] == 'fc':
                net.append(Block(d_in, *layer[1:]))
            d_in = layer[1]
        self.net = nn.ModuleList(net)
    
    def forward(self, x, y=None):
        for layer in self.net:
            x = layer(x,y)
        return x
    
        


class Encoder_vae(nn.Module):
    """
    VAE Encoder
    """
    def __init__(self, input_dim, cfg):
        """
        Parameters
        ----------
        input_dim
            input dimension
        cfg
            encoder configuration, e.g. enc_cfg = [['fc', 1024, 1, 'relu'],['fc', 10, '', '']]
        """
        super().__init__()
        h_dim = cfg[-2][1]
        self.enc = NN(input_dim, cfg[:-1])
        self.mu_enc = NN(h_dim, cfg[-1:])
        self.var_enc = NN(h_dim, cfg[-1:])
        
    def reparameterize(self, mu, var):
        return mu+var.sqrt()*torch.randn(mu.size(),device=mu.device)

    def forward(self, x, y=None):
        q = self.enc(x, y)
        mu = self.mu_enc(q, y)
        var = torch.exp(self.var_enc(q, y))+1e-6
        z = self.reparameterize(mu, var)
        return z, mu, var
                                             

    
    
 

        
        
        
   
        
       

In [ ]:
"""
Simple PeakVI-like VAE implementation for scATAC-seq data (binary peak matrix).
This is a minimal, readable PyTorch implementation inspired by PeakVI (Lopez et al.).

Features:
- Encoder producing q(z|x) (Gaussian) with optional batch covariate embedding
- Decoder mapping z (+ batch) to Bernoulli logits for peak probabilities
- ELBO loss: Bernoulli reconstruction + KL to standard normal
- Simple training loop and dataset helper for AnnData or numpy arrays

Note: This implementation focuses on clarity and usability. For production use,
consider using scvi-tools' PeakVI which includes many additional features
(e.g., library size modeling, more sophisticated priors, zero-inflation handling,
and training utilities).
"""

from typing import Optional, Tuple
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


class PeakVIDataset(Dataset):
    """Dataset wrapping a binary peak matrix (cells x peaks).

    Accepts NumPy arrays or sparse matrices (CSR/CSC). Optionally accepts
    batch labels as integers (0..n_batches-1).
    """

    def __init__(self, X, batch_index: Optional[np.ndarray] = None):
        # X: (n_cells, n_peaks) dense or sparse
        if hasattr(X, 'toarray'):
            X = X.toarray()
        X = np.asarray(X, dtype=np.float32)
        # Ensure binary in [0,1]
        X = (X > 0).astype(np.float32)
        self.X = X
        self.batch_index = None if batch_index is None else np.asarray(batch_index, dtype=np.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        if self.batch_index is None:
            return self.X[idx]
        else:
            return self.X[idx], self.batch_index[idx]


class MLPEncoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int, hidden_dims=(256, 128), n_batches: Optional[int] = None, batch_embedding_dim: int = 8):
        super().__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.n_batches = n_batches

        # If batches provided, we'll embed and concat to input
        if n_batches is not None:
            self.batch_emb = nn.Embedding(n_batches, batch_embedding_dim)
            input_dim = input_dim + batch_embedding_dim
        else:
            self.batch_emb = None

        layers = []
        last_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(last_dim, h))
            layers.append(nn.ReLU())
            last_dim = h
        self.net = nn.Sequential(*layers)

        # map to mean & logvar
        self.fc_mu = nn.Linear(last_dim, latent_dim)
        self.fc_logvar = nn.Linear(last_dim, latent_dim)

    def forward(self, x: torch.Tensor, batch_index: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        # x: (B, input_dim_orig)
        if self.batch_emb is not None:
            if batch_index is None:
                raise ValueError("n_batches specified but batch_index is None in forward")
            b_emb = self.batch_emb(batch_index)
            x = torch.cat([x, b_emb], dim=-1)
        h = self.net(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar


class Decoder(nn.Module):
    def __init__(self, latent_dim: int, output_dim: int, hidden_dims=(128, 256), n_batches: Optional[int] = None, batch_embedding_dim: int = 8):
        super().__init__()
        self.n_batches = n_batches
        if n_batches is not None:
            self.batch_emb = nn.Embedding(n_batches, batch_embedding_dim)
            latent_dim_in = latent_dim + batch_embedding_dim
        else:
            self.batch_emb = None
            latent_dim_in = latent_dim

        layers = []
        last = latent_dim_in
        for h in hidden_dims:
            layers.append(nn.Linear(last, h))
            layers.append(nn.ReLU())
            last = h
        layers.append(nn.Linear(last, output_dim))  # logits for Bernoulli
        self.net = nn.Sequential(*layers)

    def forward(self, z: torch.Tensor, batch_index: Optional[torch.Tensor] = None) -> torch.Tensor:
        if self.batch_emb is not None:
            if batch_index is None:
                raise ValueError("n_batches specified but batch_index is None in forward")
            b_emb = self.batch_emb(batch_index)
            z = torch.cat([z, b_emb], dim=-1) 
        logits = self.net(z)
        return logits


class PeakVI(nn.Module):
    """PeakVI-like variational autoencoder for binary scATAC data.

    Args:
        n_peaks: number of peaks (features)
        latent_dim: dimension of latent space
        n_batches: number of batch categories (optional)

    """

    def __init__(self, n_peaks: int, latent_dim: int = 10, n_batches: Optional[int] = None):
        super().__init__()
        self.n_peaks = n_peaks
        self.latent_dim = latent_dim
        self.n_batches = n_batches
        # Encoder maps binary input -> q(z|x)
        self.encoder = MLPEncoder(n_peaks, latent_dim, hidden_dims=(512, 256), n_batches=n_batches)
        # Decoder maps z -> Bernoulli logits per peak
        self.decoder = Decoder(latent_dim, n_peaks, hidden_dims=(256, 512), n_batches=n_batches)

    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x: torch.Tensor, batch_index: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Return reconstruction logits and posterior params.

        Returns:
            logits: (B, n_peaks) Bernoulli logits
            mu: (B, latent_dim)
            logvar: (B, latent_dim)
        """
        mu, logvar = self.encoder(x, batch_index)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(z, batch_index)
        return logits, mu, logvar

    def loss(self, x: torch.Tensor, logits: torch.Tensor, mu: torch.Tensor, logvar: torch.Tensor, kl_weight: float = 1.0) -> torch.Tensor:
        # Reconstruction: Bernoulli negative log-likelihood
        recon_loss = nn.functional.binary_cross_entropy_with_logits(logits, x, reduction='sum')
        # KL divergence between q(z|x) ~ N(mu, sigma^2) and p(z)=N(0,I)
        # closed form: 0.5 * sum( mu^2 + var - 1 - logvar )
        var = torch.exp(logvar)
        kl = 0.5 * torch.sum(mu * mu + var - 1.0 - logvar)
        # ELBO (we minimize -ELBO) -> recon + kl
        loss = recon_loss + kl_weight * kl
        # normalize by batch size to keep scale stable
        loss = loss / x.shape[0]
        return loss


# Training utilities

def train_peakvi(model: PeakVI, dataloader: DataLoader, n_epochs: int = 20, lr: float = 1e-3, device: Optional[str] = None, kl_weight: float = 1.0):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(1, n_epochs + 1):
        total_loss = 0.0
        total_recon = 0.0
        total_kl = 0.0
        n = 0
        for batch in dataloader:
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                x_np, batch_idx = batch
                x = torch.tensor(x_np, dtype=torch.float32, device=device)
                batch_idx = torch.tensor(batch_idx, dtype=torch.long, device=device)
            else:
                x = torch.tensor(batch, dtype=torch.float32, device=device)
                batch_idx = None
            opt.zero_grad()
            logits, mu, logvar = model(x, batch_idx)
            # compute components separately for logging
            recon_loss = nn.functional.binary_cross_entropy_with_logits(logits, x, reduction='sum') / x.shape[0]
            var = torch.exp(logvar)
            kl = 0.5 * torch.sum(mu * mu + var - 1.0 - logvar) / x.shape[0]
            loss = recon_loss + kl_weight * kl
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.shape[0]
            total_recon += recon_loss.item() * x.shape[0]
            total_kl += kl.item() * x.shape[0]
            n += x.shape[0]
        avg_loss = total_loss / n
        avg_recon = total_recon / n
        avg_kl = total_kl / n
        print(f"Epoch {epoch}/{n_epochs} — loss: {avg_loss:.4f}, recon: {avg_recon:.4f}, kl: {avg_kl:.4f}")





In [4]:
atac_peak.X[atac_peak.X>0]=1

In [5]:

ds = PeakVIDataset(atac_peak.X, batch_index=None)
dl = DataLoader(ds, batch_size=16, shuffle=True)

model = PeakVI(n_peaks=atac_peak.shape[1], latent_dim=16, n_batches=None)
train_peakvi(model, dl, n_epochs=10, lr=3e-3)

# After training you can obtain latent embeddings:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

with torch.no_grad():
    X_tensor = torch.as_tensor(atac_peak.X, dtype=torch.float32, device=device)
    batch_idx = torch.as_tensor(None, dtype=torch.long, device=device)
    logits, mu, logvar = model(X_tensor, batch_idx)
    z = mu.cpu().numpy()  # 可转回CPU用于可视化/下游分析
    print('Latent shape:', z.shape)


C:\Users\Administrator\AppData\Local\Temp\ipykernel_13372\2431589449.py:190: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(batch, dtype=torch.float32, device=device)


Epoch 1/10 — loss: 161204.4582, recon: 21720.1914, kl: 139484.2605
Epoch 2/10 — loss: nan, recon: nan, kl: nan


KeyboardInterrupt: 